In [2]:
import numpy as np
from scipy.optimize import linprog

def min_makespan_lp(processing_times: np.ndarray) -> dict:
  n ,m = processing_times.shape
  n_vars = n*m + 1
  C_idx = n* m
  c_obj = np.zeros(n_vars)
  c_obj[C_idx] = 1.0
  A_eq = np.zeros((n, n_vars))
  b_eq = np.ones(n)
  for i in range(n):
    for j in range(m):
      A_eq[i,i * m + j] = 1.0

  A_ub = np.zeros((m , n_vars))
  b_ub = np.zeros(m)
  for j in range(m):
    for i in range(n):
      A_ub[j, i * m + j] = processing_times[i,j]
    A_ub[j, C_idx] = -1.0

  bounds = [(0.0, 1.0)] * (n * m) + [(0.0, None)]

  result = linprog(c_obj, A_ub= A_ub, b_ub=b_ub,
                   A_eq=A_eq, b_eq=b_eq,
                   bounds = bounds,
                   method="highs")

  if result.status  != 0:
      raise RuntimeError(f"LP fallo: {result.message}")

  x_flat = result.x[:n*m]
  x   = x_flat.reshape(n,m)
  C_opt = result.x[C_idx]
  loads = np.array([np.sum(processing_times[:,j] * x[:,j]) for j in range(m)])
  return {"x": x, "makespan": C_opt, "loads": loads}





def min_cost_lp(job_loads: np.ndarray,
                server_costs: np.ndarray,
                server_caps: np.ndarray) -> dict:

                n = len(job_loads)
                m = len(server_costs)
                n_vars = n * m

                c_obj = np.array([server_costs[j]* job_loads[i]
                                  for i in range(n) for j in range(m)])
                A_eq = np.zeros((n,n_vars))
                b_eq = np.ones(n)
                for i in range(n):
                  for j in range(m):
                    A_eq[i, i*m + j] = 1.0

                A_ub = np.zeros((m,n_vars))
                b_ub = server_caps.copy().astype(float)
                for j in range(m):
                  # Fixed: Loop over i for A_ub definition
                  for i in range(n):
                    A_ub[j,i * m + j] = job_loads[i]

                bounds = [(0.0, 1.0)] * n_vars

                result = linprog(c_obj, A_ub= A_ub, b_ub=b_ub,
                                 A_eq=A_eq,b_eq=b_eq,
                                 bounds=bounds, method="highs")

                if result.status != 0:
                  raise RuntimeError(f"LP fallo: {result.message}")

                x = result.x.reshape(n,m)
                loads = np.array([np.sum(job_loads * x[:,j]) for j in range(m)])
                total_cost = float(result.fun)

                return{"x": x, "total_cost": total_cost, "loads": loads}


def traffic_balancing_lp(request_rates: np.ndarray,
                         latency_matrix: np.ndarray,
                         sla_limits: np.ndarray) -> dict:

    k = len(request_rates)
    m = latency_matrix.shape[1]
    n_vars = k * m + 1
    C_idx = k * m

    feasible = latency_matrix <= sla_limits[:, None]
    c_obj = np.zeros(n_vars)
    c_obj[C_idx] = 1.0
    A_eq = np.zeros((k,n_vars))
    b_eq = np.ones(k)
    for i in range(k):
      for j in range(m):
        A_eq[i, i*m + j] = 1.0

    A_ub = np.zeros((m,n_vars))
    b_ub = np.zeros(m)
    for j in range(m):
      for i in range(k):
        A_ub[j, i * m + j] = request_rates[i]
      A_ub[j, C_idx] = -1.0

    bounds = []
    for i in range(k):
        for j in range(m):
          ub = 1.0 if feasible[i,j] else 0.0
          bounds.append((0.0, ub))
    bounds.append((0.0, None))

    result = linprog(c_obj, A_ub=A_ub, b_ub=b_ub,
                     A_eq=A_eq, b_eq=b_eq,
                     bounds=bounds, method="highs")
    if result.status != 0:
      raise RuntimeError(f"LP fallo: {result.message}")

    x = result.x[:k * m].reshape(k,m)
    max_load = result.x[C_idx]
    loads = np.array([np.sum(request_rates * x[:,j]) for j in range(m)])


    return {"x":x, "max_load": max_load, "loads": loads,
            "feasible_mask": feasible}


def demo():
    sep = "=" * 62

    # ── DEMO 1: Min-Makespan ──────────────────────────────────────────────────
    print(sep)
    print(" MODELO 1: Min-Makespan LP")
    print(sep)

    # 5 trabajos × 3 servidores: p[i][j] = tiempo del trabajo i en servidor j
    P = np.array([
        [3, 5, 2],   # trabajo 0
        [4, 1, 6],   # trabajo 1
        [2, 3, 4],   # trabajo 2
        [5, 2, 1],   # trabajo 3
        [1, 4, 3],   # trabajo 4
    ], dtype=float)

    res1 = min_makespan_lp(P)

    print(f"\nMatriz de tiempos p[trabajo][servidor]:\n{P}\n")
    print("Asignación LP (fracciones):")
    for i, row in enumerate(res1["x"]):
        assigned = {f"S{j}": f"{v:.2f}" for j, v in enumerate(row) if v > 1e-4}
        print(f"  Trabajo {i}: {assigned}")
    print(f"\nCarga por servidor: {[f'{l:.2f}' for l in res1['loads']]}")
    print(f"Makespan (C*):      {res1['makespan']:.4f}")

    # ── DEMO 2: Costo Mínimo ──────────────────────────────────────────────────
    print(f"\n{sep}")
    print(" MODELO 2: Costo Mínimo con Capacidades")
    print(sep)

    job_loads    = np.array([10, 20, 15, 8, 25], dtype=float)   # unidades
    server_costs = np.array([1.0, 2.5, 1.8], dtype=float)        # $/unidad
    server_caps  = np.array([40, 35, 30], dtype=float)            # cap máx

    res2 = min_cost_lp(job_loads, server_costs, server_caps)

    print(f"\nCargas de trabajos : {job_loads}")
    print(f"Costos de servidores: {server_costs}  ($/unidad)")
    print(f"Capacidades         : {server_caps}")
    print("\nAsignación LP (fracciones):")
    for i, row in enumerate(res2["x"]):
        assigned = {f"S{j}": f"{v:.2f}" for j, v in enumerate(row) if v > 1e-4}
        print(f"  Trabajo {i} (carga={job_loads[i]}): {assigned}")
    print(f"\nCarga por servidor : {[f'{l:.2f}' for l in res2['loads']]}")
    print(f"Costo total óptimo : {res2['total_cost']:.4f}")

    # ── DEMO 3: Tráfico + SLA ─────────────────────────────────────────────────
    print(f"\n{sep}")
    print(" MODELO 3: Balanceo de Tráfico con SLA")
    print(sep)

    # 4 tipos de tráfico, 3 servidores
    rates   = np.array([100, 200, 150, 80], dtype=float)  # req/s
    latency = np.array([                                   # ms
        [ 5, 12, 20],   # tipo 0 (API crítica)
        [15,  8, 25],   # tipo 1 (web estática)
        [30, 10,  6],   # tipo 2 (batch)
        [ 8, 18, 12],   # tipo 3 (streaming)
    ], dtype=float)
    sla = np.array([10, 20, 15, 20], dtype=float)  # ms máx por tipo

    res3 = traffic_balancing_lp(rates, latency, sla)

    print(f"\nTasas de tráfico (req/s): {rates}")
    print(f"SLA por tipo (ms)       : {sla}")
    print("\nMatriz de latencia (ms):")
    header = "           " + "  ".join(f"S{j:>4}" for j in range(latency.shape[1]))
    print(header)
    for i, row in enumerate(latency):
        cells = "  ".join(f"{v:5.0f}{'(*)' if v > sla[i] else '   '}"
                          for j_, v in enumerate(row))
        print(f"  Tipo {i} SLA={sla[i]:4.0f}ms: {cells}")
    print("  (*) = ruta bloqueada por SLA")

    print("\nAsignación de tráfico (fracciones):")
    for i, row in enumerate(res3["x"]):
        assigned = {f"S{j}": f"{v:.2f}" for j, v in enumerate(row) if v > 1e-4}
        print(f"  Tipo {i} ({rates[i]} req/s): {assigned}")
    print(f"\nCarga por servidor (req/s): {[f'{l:.1f}' for l in res3['loads']]}")
    print(f"Carga máxima óptima (C*)  : {res3['max_load']:.4f} req/s")




if __name__ == "__main__":
    demo()


 MODELO 1: Min-Makespan LP

Matriz de tiempos p[trabajo][servidor]:
[[3. 5. 2.]
 [4. 1. 6.]
 [2. 3. 4.]
 [5. 2. 1.]
 [1. 4. 3.]]

Asignación LP (fracciones):
  Trabajo 0: {'S2': '1.00'}
  Trabajo 1: {'S1': '1.00'}
  Trabajo 2: {'S0': '0.78', 'S1': '0.22'}
  Trabajo 3: {'S1': '0.44', 'S2': '0.56'}
  Trabajo 4: {'S0': '1.00'}

Carga por servidor: ['2.56', '2.56', '2.56']
Makespan (C*):      2.5556

 MODELO 2: Costo Mínimo con Capacidades

Cargas de trabajos : [10. 20. 15.  8. 25.]
Costos de servidores: [1.  2.5 1.8]  ($/unidad)
Capacidades         : [40. 35. 30.]

Asignación LP (fracciones):
  Trabajo 0 (carga=10.0): {'S2': '1.00'}
  Trabajo 1 (carga=20.0): {'S2': '1.00'}
  Trabajo 2 (carga=15.0): {'S0': '1.00'}
  Trabajo 3 (carga=8.0): {'S1': '1.00'}
  Trabajo 4 (carga=25.0): {'S0': '1.00'}

Carga por servidor : ['40.00', '8.00', '30.00']
Costo total óptimo : 114.0000

 MODELO 3: Balanceo de Tráfico con SLA

Tasas de tráfico (req/s): [100. 200. 150.  80.]
SLA por tipo (ms)       : [10. 